# 🎯 ESG RAG 공동작업: Google Colab 버전 (OCR 지원)

**목표**: 100개사 PDF → 단일 검색 인덱스 생성 (Gemini API 사용)

**특징**: 텍스트 PDF + 이미지 PDF 모두 지원

**소요시간**: ~2시간 (GPU 활용)

## 📋 실행 순서
1. ✅ 라이브러리 설치 (OCR 포함)
2. ✅ 라이브러리 import
3. ✅ Gemini API 키 설정
4. ✅ Google Drive 마운트
5. ✅ 데이터 준비
6. ✅ Step 1-5 순차 실행 (텍스트/이미지 PDF 모두 처리)
7. ✅ 결과 확인

## 셀 1️⃣: 라이브러리 설치 (OCR 포함)

In [2]:
# Colab 시스템 패키지: Tesseract OCR 설치
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

# Python 패키지 설치
!pip install -q pdfplumber chromadb langchain-chroma langchain-community \
    langchain-google-genai kiwipiepy rank_bm25 pandas tqdm \
    pytesseract pdf2image pillow

print("\n✅ 라이브러리 설치 완료!")
print("\n📦 설치된 패키지:")
print("  - pdfplumber (텍스트 PDF 파싱)")
print("  - pdf2image (PDF → 이미지 변환)")
print("  - pytesseract (OCR 텍스트 추출)")
print("  - langchain-chroma (벡터DB)")
print("  - langchain-google-genai (Gemini 임베딩)")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 11

## 셀 2️⃣: 라이브러리 import

In [3]:
# 데이터 처리
import pandas as pd
import json
import os
from pathlib import Path
from datetime import datetime
from typing import List, Dict

# PDF 처리 (텍스트)
import pdfplumber

# PDF 처리 (이미지 + OCR)
from pdf2image import convert_from_path
import pytesseract
from PIL import Image

# 벡터DB
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 진행 표시
from tqdm import tqdm

# BM25 (한국어 검색)
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

print("✅ 모든 라이브러리 import 완료!")
print("\n📚 활성화된 모듈:")
print("  ✓ pdfplumber (텍스트 PDF)")
print("  ✓ pdf2image (이미지 변환)")
print("  ✓ pytesseract (OCR)")
print("  ✓ langchain (벡터DB 및 임베딩)")

✅ 모든 라이브러리 import 완료!

📚 활성화된 모듈:
  ✓ pdfplumber (텍스트 PDF)
  ✓ pdf2image (이미지 변환)
  ✓ pytesseract (OCR)
  ✓ langchain (벡터DB 및 임베딩)


## 셀 3️⃣: Gemini API 키 설정

In [4]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Colab 시크릿에서 API 키 로드됨")
except:
    print("⚠️ Colab 시크릿 없음")
    print("\n다음 단계를 따르세요:")
    print("1. 왼쪽 패널에서 '🔑' 아이콘 클릭")
    print("2. '새 시크릿 추가' 클릭")
    print("3. 이름: GEMINI_API_KEY")
    print("4. 값: sk-...")
    print("5. 저장")
    GEMINI_API_KEY = input("\nAPI 키를 입력하세요: ")

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
print(f"\n✅ API 키 설정 완료 (길이: {len(GEMINI_API_KEY)}자)")

⚠️ Colab 시크릿 없음

다음 단계를 따르세요:
1. 왼쪽 패널에서 '🔑' 아이콘 클릭
2. '새 시크릿 추가' 클릭
3. 이름: GEMINI_API_KEY
4. 값: sk-...
5. 저장

API 키를 입력하세요: AQ.Ab8RN6KYL9iDCbVyMoQ8PjKPpPmL7hcKPvT2xgRcdeOdpXwbGw

✅ API 키 설정 완료 (길이: 53자)


## 셀 4️⃣: Google Drive 마운트

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# 작업 디렉토리 설정
work_dir = Path('/content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)')
work_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Google Drive 마운트 완료")
print(f"   작업 디렉토리: {work_dir}")

✅ Google Drive 마운트 완료
   작업 디렉토리: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)


## 셀 5️⃣: Manifest 파일 준비

In [9]:
# 샘플 manifest.csv (LS일렉트릭 테스트용)

manifest_data = {
    'company_code': ['010120', '010120', '010120'],
    'company_name': ['LS일렉트릭', 'LS일렉트릭', 'LS일렉트릭'],
    'year': [2025, 2025, 2025],
    'doc_type': [
        'business',           # 사업보고서
        'governance',         # 지배구조보고서
        'sustainability'      # 지속가능경영보고서
    ],
    'file_path': [
        '/content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/010120_2025_business.pdf',
        '/content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/010120_2025_governance.pdf',
        '/content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/010120_2025_sustainability.pdf'
    ]
}

df_manifest = pd.DataFrame(manifest_data)
manifest_path = work_dir / 'manifest.csv'
df_manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')

print("✅ manifest.csv 생성 완료")
print("\n📋 내용:")
print(df_manifest)
print(f"\n💾 저장 위치: {manifest_path}")

✅ manifest.csv 생성 완료

📋 내용:
  company_code company_name  year        doc_type  \
0       010120       LS일렉트릭  2025        business   
1       010120       LS일렉트릭  2025      governance   
2       010120       LS일렉트릭  2025  sustainability   

                                           file_path  
0  /content/drive/MyDrive/Colab Notebooks/2026_DA...  
1  /content/drive/MyDrive/Colab Notebooks/2026_DA...  
2  /content/drive/MyDrive/Colab Notebooks/2026_DA...  

💾 저장 위치: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/manifest.csv


## 셀 6️⃣: 파일 존재 확인

In [10]:
print("📂 PDF 파일 확인 중...\n")

missing_files = []

for idx, row in df_manifest.iterrows():
    file_path = work_dir / row['file_path']

    if file_path.exists():
        size = file_path.stat().st_size / (1024*1024)
        print(f"✅ {row['company_name']} ({row['doc_type']}): {size:.1f} MB")
    else:
        print(f"❌ 없음: {file_path}")
        missing_files.append(file_path)

if missing_files:
    print(f"\n⚠️ {len(missing_files)}개 파일 누락!")
    print("\n해결 방법:")
    print("1. Google Drive에 /ESG_RAG_Project/pdfs/ 폴더 생성")
    print("2. 해당 폴더에 PDF 파일 업로드")
    print("3. 위 셀 다시 실행")
else:
    print(f"\n✅ 모든 파일 준비 완료! 다음 단계로 진행 가능")

📂 PDF 파일 확인 중...

✅ LS일렉트릭 (business): 1.6 MB
✅ LS일렉트릭 (governance): 2.7 MB
✅ LS일렉트릭 (sustainability): 16.5 MB

✅ 모든 파일 준비 완료! 다음 단계로 진행 가능


## 셀 7️⃣: Step 1 - PDF 파싱 (텍스트 + OCR)

In [11]:
# ============================================================================
# 파싱 조건 설정
# ============================================================================

PARSING_CONFIG = {
    'use_ocr': True,                    # OCR 사용 여부
    'min_text_threshold': 100,          # 최소 텍스트 길이 (자 이상)
    'max_ocr_pages': 50,                # OCR 처리 최대 페이지 수
    'ocr_confidence_threshold': 0.3     # OCR 신뢰도 최소값
}

print("⚙️ 파싱 설정:")
print(f"  - OCR 사용: {PARSING_CONFIG['use_ocr']}")
print(f"  - 최소 텍스트: {PARSING_CONFIG['min_text_threshold']}글자")
print(f"  - 최대 OCR 페이지: {PARSING_CONFIG['max_ocr_pages']}페이지")
print(f"  - OCR 신뢰도 최소: {PARSING_CONFIG['ocr_confidence_threshold']:.1%}")

print(f"\n📖 파싱 로직:")
print(f"  1️⃣ pdfplumber로 텍스트 추출")
print(f"  2️⃣ 텍스트 길이 < {PARSING_CONFIG['min_text_threshold']}글자? → OCR 수행")
print(f"  3️⃣ 페이지 수 > {PARSING_CONFIG['max_ocr_pages']}? → OCR 스킵 (너무 많음)")
print(f"  4️⃣ 결합 결과 저장")

⚙️ 파싱 설정:
  - OCR 사용: True
  - 최소 텍스트: 100글자
  - 최대 OCR 페이지: 50페이지
  - OCR 신뢰도 최소: 30.0%

📖 파싱 로직:
  1️⃣ pdfplumber로 텍스트 추출
  2️⃣ 텍스트 길이 < 100글자? → OCR 수행
  3️⃣ 페이지 수 > 50? → OCR 스킵 (너무 많음)
  4️⃣ 결합 결과 저장


## 셀 8️⃣: Step 1 - PDF 파싱 함수 (텍스트 + OCR)

In [12]:
def parse_pdf_with_ocr(pdf_path: str, config: Dict) -> List[Dict]:
    """
    PDF 파싱 함수 (텍스트 + 이미지 PDF 모두 지원)

    입력: PDF 경로
    출력: 페이지별 데이터 리스트
    """
    pages_data = []
    pdf_path = Path(pdf_path)

    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            total_pages = len(pdf.pages)
            ocr_used = False
            ocr_pages = 0

            for page_num, page in enumerate(pdf.pages, start=1):
                # (1) pdfplumber로 텍스트 추출
                text = page.extract_text() or ""

                # (2) 표 추출
                tables = page.extract_tables() or []
                tables_markdown = ""
                for table in tables:
                    if not table:
                        continue
                    header = [str(cell or "").replace("\n", " ") for cell in table[0]]
                    tables_markdown += "| " + " | ".join(header) + " |\n"
                    tables_markdown += "|" + "|".join(["---"] * len(header)) + "|\n"
                    for row in table[1:]:
                        cells = [str(cell or "").replace("\n", " ") for cell in row]
                        tables_markdown += "| " + " | ".join(cells) + " |\n"
                    tables_markdown += "\n"

                # (3) OCR 필요성 판단
                needs_ocr = (
                    config['use_ocr'] and
                    len(text.strip()) < config['min_text_threshold'] and
                    ocr_pages < config['max_ocr_pages']  # OCR 페이지 수 제한
                )

                # (4) OCR 수행
                ocr_text = ""
                if needs_ocr:
                    try:
                        # PDF 페이지를 이미지로 변환
                        images = convert_from_path(str(pdf_path), first_page=page_num, last_page=page_num)
                        if images:
                            # Tesseract OCR로 텍스트 추출
                            ocr_text = pytesseract.image_to_string(images[0], lang='kor+eng')
                            ocr_used = True
                            ocr_pages += 1
                    except Exception as e:
                        ocr_text = ""  # OCR 실패 시 빈 문자열

                # (5) 텍스트 결합
                combined_text = text
                if ocr_text.strip():
                    combined_text = text + "\n[OCR 추출]\n" + ocr_text

                # (6) 섹션 제목 추출
                lines = text.split("\n")[:3]
                section = " / ".join([l.strip() for l in lines if l.strip()])[:50]

                # (7) 페이지 저장
                if combined_text.strip() or tables:
                    pages_data.append({
                        "page": page_num,
                        "section": section,
                        "text": text,
                        "ocr_text": ocr_text,
                        "tables_markdown": tables_markdown,
                        "combined": combined_text,
                        "parsing_method": "ocr" if ocr_text.strip() else "text"
                    })

    except Exception as e:
        print(f"❌ 오류: {e}")
        return []

    return pages_data, ocr_used, ocr_pages

print("✅ parse_pdf_with_ocr 함수 정의 완료")

✅ parse_pdf_with_ocr 함수 정의 완료


## 셀 9️⃣: Step 1 실행 - PDF 파싱 (텍스트 + OCR)

In [13]:
print("📖 Step 1: PDF 파싱 중...\n")

all_pages = {}
ocr_stats = {}

for idx, row in df_manifest.iterrows():
    pdf_path = work_dir / row['file_path']
    company = row['company_name']
    doc_type = row['doc_type']
    key = f"{company}_{doc_type}"

    print(f"처리 중: {company} - {doc_type}")

    pages, ocr_used, ocr_count = parse_pdf_with_ocr(str(pdf_path), PARSING_CONFIG)

    # 파싱 방법 통계
    text_only = sum(1 for p in pages if p['parsing_method'] == 'text')
    ocr_only = sum(1 for p in pages if p['parsing_method'] == 'ocr')

    print(f"  ✅ {len(pages)}페이지 추출됨")
    print(f"     - 텍스트 기반: {text_only}페이지")
    print(f"     - OCR 기반: {ocr_only}페이지")
    if ocr_used:
        print(f"     - OCR 사용: {ocr_count}페이지")
    print()

    all_pages[key] = pages
    ocr_stats[key] = {'text_pages': text_only, 'ocr_pages': ocr_only, 'total': len(pages)}

total_pages = sum(len(p) for p in all_pages.values())
total_ocr = sum(s['ocr_pages'] for s in ocr_stats.values())

print(f"\n✅ Step 1 완료!")
print(f"\n📊 파싱 통계:")
print(f"  - 총 페이지: {total_pages}개")
print(f"  - 텍스트 기반: {total_pages - total_ocr}개")
print(f"  - OCR 기반: {total_ocr}개 ({100*total_ocr/total_pages:.1f}%)")

📖 Step 1: PDF 파싱 중...

처리 중: LS일렉트릭 - business
  ✅ 435페이지 추출됨
     - 텍스트 기반: 435페이지
     - OCR 기반: 0페이지

처리 중: LS일렉트릭 - governance


  ✅ 50페이지 추출됨
     - 텍스트 기반: 50페이지
     - OCR 기반: 0페이지

처리 중: LS일렉트릭 - sustainability
  ✅ 101페이지 추출됨
     - 텍스트 기반: 101페이지
     - OCR 기반: 0페이지


✅ Step 1 완료!

📊 파싱 통계:
  - 총 페이지: 586개
  - 텍스트 기반: 586개
  - OCR 기반: 0개 (0.0%)


## 셀 🔟: Step 2 - 청킹

In [14]:
def create_chunks(pages_data: List[Dict], chunk_size: int = 800, overlap: int = 150) -> List[Dict]:
    """
    페이지 텍스트를 청크로 분할
    """
    chunks = []
    chunk_id = 0

    for page_data in pages_data:
        text = page_data["combined"]
        page_num = page_data["page"]
        section = page_data["section"]
        parsing_method = page_data.get("parsing_method", "text")

        for i in range(0, len(text), chunk_size - overlap):
            chunk_text = text[i:i + chunk_size].strip()

            if len(chunk_text) < 100:
                continue

            chunks.append({
                "chunk_id": f"chunk{chunk_id:05d}",
                "page": page_num,
                "section": section,
                "text": chunk_text,
                "char_count": len(chunk_text),
                "parsing_method": parsing_method
            })
            chunk_id += 1

    return chunks

print("✂️ Step 2: 청킹 중...\n")

all_chunks = []
chunk_stats = {}

for pages_key, pages in all_pages.items():
    chunks = create_chunks(pages, chunk_size=800, overlap=150)
    all_chunks.extend(chunks)

    # 청킹 방법 통계
    text_chunks = sum(1 for c in chunks if c['parsing_method'] == 'text')
    ocr_chunks = sum(1 for c in chunks if c['parsing_method'] == 'ocr')

    print(f"{pages_key}: {len(chunks)}개 청크")
    print(f"  - 텍스트 기반: {text_chunks}개")
    print(f"  - OCR 기반: {ocr_chunks}개")
    chunk_stats[pages_key] = {'text': text_chunks, 'ocr': ocr_chunks, 'total': len(chunks)}

print(f"\n✅ Step 2 완료! 총 {len(all_chunks)}개 청크 생성됨")
print(f"\n📊 청크 크기 통계:")
sizes = [c['char_count'] for c in all_chunks]
print(f"  - 평균: {sum(sizes)/len(sizes):.0f}글자")
print(f"  - 최대: {max(sizes)}글자")
print(f"  - 최소: {min(sizes)}글자")

# 파싱 방법별 청크
total_text_chunks = sum(s['text'] for s in chunk_stats.values())
total_ocr_chunks = sum(s['ocr'] for s in chunk_stats.values())
print(f"\n📈 파싱 방법별 청크:")
print(f"  - 텍스트 기반: {total_text_chunks}개 ({100*total_text_chunks/len(all_chunks):.1f}%)")
print(f"  - OCR 기반: {total_ocr_chunks}개 ({100*total_ocr_chunks/len(all_chunks):.1f}%)")

✂️ Step 2: 청킹 중...

LS일렉트릭_business: 962개 청크
  - 텍스트 기반: 962개
  - OCR 기반: 0개
LS일렉트릭_governance: 162개 청크
  - 텍스트 기반: 162개
  - OCR 기반: 0개
LS일렉트릭_sustainability: 244개 청크
  - 텍스트 기반: 244개
  - OCR 기반: 0개

✅ Step 2 완료! 총 1368개 청크 생성됨

📊 청크 크기 통계:
  - 평균: 634글자
  - 최대: 800글자
  - 최소: 100글자

📈 파싱 방법별 청크:
  - 텍스트 기반: 1368개 (100.0%)
  - OCR 기반: 0개 (0.0%)


## 셀 1️⃣1️⃣: Step 3 - 메타데이터

In [15]:
print("📝 Step 3: 메타데이터 생성 중...\n")

docs_with_metadata = []

for idx, row in df_manifest.iterrows():
    company_code = row['company_code']
    company_name = row['company_name']
    year = int(row['year'])
    doc_type = row['doc_type']
    pages_key = f"{company_name}_{doc_type}"

    if pages_key not in all_pages:
        print(f"⚠️ {pages_key} 데이터 없음")
        continue

    chunks = create_chunks(all_pages[pages_key])

    for chunk in chunks:
        doc = Document(
            page_content=chunk['text'],
            metadata={
                'company_code': company_code,
                'company_name': company_name,
                'year': year,
                'doc_type': doc_type,
                'doc_id': f"{company_code}_{year}_{doc_type}",
                'chunk_id': chunk['chunk_id'],
                'page': chunk['page'],
                'section': chunk['section'],
                'content_type': 'table' if '|' in chunk['text'] else 'text',
                'char_count': chunk['char_count'],
                'parsing_method': chunk['parsing_method'],  # ← 파싱 방법 기록
                'indexed_at': datetime.now().isoformat()
            }
        )
        docs_with_metadata.append(doc)

    print(f"✅ {company_name} ({doc_type}): {len(chunks)}개 청크")

print(f"\n✅ Step 3 완료! 메타데이터 생성됨")
print(f"\n📊 메타데이터 통계:")
print(f"  - 총 청크: {len(docs_with_metadata)}개")
print(f"  - 텍스트 파싱: {sum(1 for d in docs_with_metadata if d.metadata.get('parsing_method') == 'text')}개")
print(f"  - OCR 파싱: {sum(1 for d in docs_with_metadata if d.metadata.get('parsing_method') == 'ocr')}개")

📝 Step 3: 메타데이터 생성 중...

✅ LS일렉트릭 (business): 962개 청크
✅ LS일렉트릭 (governance): 162개 청크
✅ LS일렉트릭 (sustainability): 244개 청크

✅ Step 3 완료! 메타데이터 생성됨

📊 메타데이터 통계:
  - 총 청크: 1368개
  - 텍스트 파싱: 1368개
  - OCR 파싱: 0개


## 셀 1️⃣2️⃣: Step 4 - Gemini 벡터 임베딩

In [16]:
print("🧠 Step 4: Gemini 벡터 임베딩 중...")
print(f"   (청크 개수: {len(docs_with_metadata)}개)\n")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)
print("✅ Gemini 임베딩 모델 로드 완료\n")

index_dir = work_dir / "unified_index"
chroma_dir = index_dir / "chroma_db"
index_dir.mkdir(parents=True, exist_ok=True)
chroma_dir.mkdir(parents=True, exist_ok=True)

print(f"📚 Chroma 벡터DB 구축 중...")
print(f"   저장 위치: {chroma_dir}\n")

batch_size = 50
vectorstore = None

for batch_start in tqdm(range(0, len(docs_with_metadata), batch_size),
                        desc="벡터 임베딩",
                        unit="batch"):
    batch_end = min(batch_start + batch_size, len(docs_with_metadata))
    batch_docs = docs_with_metadata[batch_start:batch_end]

    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch_docs,
            embedding=embeddings,
            persist_directory=str(chroma_dir)
        )
    else:
        vectorstore.add_documents(batch_docs)

print(f"\n✅ Step 4 완료! 벡터 임베딩 완료")

🧠 Step 4: Gemini 벡터 임베딩 중...
   (청크 개수: 1368개)

✅ Gemini 임베딩 모델 로드 완료

📚 Chroma 벡터DB 구축 중...
   저장 위치: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/unified_index/chroma_db



벡터 임베딩: 100%|██████████| 28/28 [01:22<00:00,  2.94s/batch]


✅ Step 4 완료! 벡터 임베딩 완료


## 셀 1️⃣3️⃣: Step 5 - 결과 저장

In [17]:
print("💾 Step 5: 결과 저장 중...\n")

metadata_list = [
    {
        'chunk_id': doc.metadata['chunk_id'],
        'company_code': doc.metadata['company_code'],
        'company_name': doc.metadata['company_name'],
        'year': doc.metadata['year'],
        'doc_type': doc.metadata['doc_type'],
        'page': doc.metadata['page'],
        'section': doc.metadata['section'],
        'content_type': doc.metadata['content_type'],
        'char_count': doc.metadata['char_count'],
        'parsing_method': doc.metadata['parsing_method'],  # ← 파싱 방법 저장
        'indexed_at': doc.metadata['indexed_at']
    }
    for doc in docs_with_metadata
]

df_metadata = pd.DataFrame(metadata_list)
metadata_csv = index_dir / "index_metadata.csv"
df_metadata.to_csv(metadata_csv, index=False, encoding='utf-8-sig')

print(f"✅ 메타데이터 저장: {metadata_csv}")

stats = {
    'total_documents': len(docs_with_metadata),
    'total_chunks': len(docs_with_metadata),
    'companies': int(df_metadata['company_code'].nunique()),
    'years': df_metadata['year'].unique().tolist(),
    'doc_types': df_metadata['doc_type'].unique().tolist(),
    'avg_chunk_size': int(df_metadata['char_count'].mean()),
    'parsing_method_text': int((df_metadata['parsing_method'] == 'text').sum()),
    'parsing_method_ocr': int((df_metadata['parsing_method'] == 'ocr').sum()),
    'embedding_model': 'gemini-embedding-001',
    'created_at': datetime.now().isoformat()
}

stats_json = index_dir / "index_stats.json"
with open(stats_json, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"✅ 통계 저장: {stats_json}")
print(f"\n📊 인덱스 통계:")
print(f"  - 총 청크: {stats['total_chunks']:,}개")
print(f"  - 회사: {stats['companies']}개")
print(f"  - 텍스트 파싱: {stats['parsing_method_text']}개 ({100*stats['parsing_method_text']/stats['total_chunks']:.1f}%)")
print(f"  - OCR 파싱: {stats['parsing_method_ocr']}개 ({100*stats['parsing_method_ocr']/stats['total_chunks']:.1f}%)")

💾 Step 5: 결과 저장 중...

✅ 메타데이터 저장: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/unified_index/index_metadata.csv
✅ 통계 저장: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/unified_index/index_stats.json

📊 인덱스 통계:
  - 총 청크: 1,368개
  - 회사: 1개
  - 텍스트 파싱: 1368개 (100.0%)
  - OCR 파싱: 0개 (0.0%)


## 셀 1️⃣4️⃣: 검색 테스트

In [18]:
print("🔍 인덱스 검색 테스트\n")

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = Chroma(
    persist_directory=str(chroma_dir),
    embedding_function=embeddings
)

test_queries = [
    "재생에너지 RE100",
    "온실가스 감축",
]

for query in test_queries:
    print(f"🔎 쿼리: '{query}'")

    results = vectorstore.similarity_search_with_score(
        query,
        k=3
    )

    if results:
        for i, (doc, score) in enumerate(results, 1):
            parsing_method = doc.metadata.get('parsing_method', 'unknown')
            print(f"  {i}. [{doc.metadata['doc_type']}] p.{doc.metadata['page']} ({parsing_method})")
            print(f"     유사도: {1-score:.2%}")
            print(f"     내용: {doc.page_content[:80]}...\n")
    else:
        print(f"  검색 결과 없음\n")

print("✅ 검색 테스트 완료!")

🔍 인덱스 검색 테스트

🔎 쿼리: '재생에너지 RE100'
  1. [sustainability] p.38 (unknown)
     유사도: 59.87%
     내용: 추진
2 배출량의 비율이 높은 당사 특성에 따라 재생에너지 조달 실행력 · RE100 가입 및 이행 활동 공급 계약(PPA) 체결
을 강화하고,...

  2. [sustainability] p.38 (text)
     유사도: 59.22%
     내용: 추진
2 배출량의 비율이 높은 당사 특성에 따라 재생에너지 조달 실행력 · RE100 가입 및 이행 활동 공급 계약(PPA) 체결
을 강화하고,...

  3. [sustainability] p.39 (text)
     유사도: 56.93%
     내용: 로도 자가발전 설비 투자 확대, 전력 공급 계약(PPA) 체결 등 다양한 방안을 적극적으로
추진하여 2040년까지 전력 공급의 100%를 지속가...

🔎 쿼리: '온실가스 감축'
  1. [sustainability] p.38 (unknown)
     유사도: 48.21%
     내용: About LS ELECTRIC ESG Management ESG Performance Appendix 2024-2025 LS ELECTRIC ...

  2. [sustainability] p.38 (text)
     유사도: 48.21%
     내용: About LS ELECTRIC ESG Management ESG Performance Appendix 2024-2025 LS ELECTRIC ...

  3. [sustainability] p.39 (unknown)
     유사도: 48.16%
     내용: 정책 개선과 시장 성장을
LS ELECTRIC은 탄소중립 실현을 위해 노후 설비 교체를 통한 에너지 효율 최적화, 고효율 인버터 설치
선도하며,...

✅ 검색 테스트 완료!


## 셀 1️⃣5️⃣: 🎉 공동작업 완료!

In [19]:
print("="*80)
print("✅ 공동작업 완료!")
print("="*80)

print(f"\n📊 최종 결과:")
print(f"  - 처리한 PDF: {len(df_manifest)}개")
print(f"  - 생성된 청크: {len(docs_with_metadata):,}개")
print(f"    • 텍스트 파싱: {stats['parsing_method_text']}개")
print(f"    • OCR 파싱: {stats['parsing_method_ocr']}개")
print(f"  - 저장 위치: {index_dir}")

print(f"\n📁 생성된 파일:")
print(f"  1. chroma_db/          ← Gemini 벡터 임베딩")
print(f"  2. index_metadata.csv  ← 메타데이터 (파싱 방법 포함)")
print(f"  3. index_stats.json    ← 통계")

print(f"\n🎯 다음 단계:")
print(f"  1. E_PART_02_COLAB.ipynb 실행")
print(f"  2. 인덱스 경로: {index_dir}")
print(f"  3. E 환경 항목별 자동 채점")
print(f"  4. 0~100 점수 산정")

print(f"\n" + "="*80)
print(f"완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

✅ 공동작업 완료!

📊 최종 결과:
  - 처리한 PDF: 3개
  - 생성된 청크: 1,368개
    • 텍스트 파싱: 1368개
    • OCR 파싱: 0개
  - 저장 위치: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/unified_index

📁 생성된 파일:
  1. chroma_db/          ← Gemini 벡터 임베딩
  2. index_metadata.csv  ← 메타데이터 (파싱 방법 포함)
  3. index_stats.json    ← 통계

🎯 다음 단계:
  1. E_PART_02_COLAB.ipynb 실행
  2. 인덱스 경로: /content/drive/MyDrive/Colab Notebooks/2026_DA_Project/02. ESG_Consulting/엘에스일렉트릭(010120)/unified_index
  3. E 환경 항목별 자동 채점
  4. 0~100 점수 산정

완료 시각: 2026-09-02 02:54:58
